In [1]:
import csv
from datetime import date

# -------------------------------------------------------------------------
# 1) DEFINE YOUR DEMOGRAPHIC GROUPS
#    Adjust groups to match how you collect data (age ranges, gender, etc.)
# -------------------------------------------------------------------------
DEMOGRAPHIC_GROUPS = [
    "18-25_male",
    "26-40_male",
    "18-25_female",
    "26-40_female",
    # add more groups if needed
]

# -------------------------------------------------------------------------
# 2) TEMPLATE FIELDS FOR EACH ROW
# -------------------------------------------------------------------------
FIELD_NAMES = [
    # basic identifiers
    "date",                # YYYY-MM-DD
    "country",             # UK or US
    "region_or_city",      # optional, e.g., postcode area or state
    "sku",                 # product SKU/name
    "flavor",              # Watermelon
    "servings_per_tub",    # 50
    
    # sales metrics
    "units_sold",
    "revenue_local",       # in local currency, e.g., GBP or USD
    "cost_of_goods_sold",  # actual purchase cost or estimate
    "shipping_cost",       # per order or average
    "platform_fees",       # e.g., marketplace fees, card fees
    "other_expenses",      # marketing, returns, etc.
    "gross_profit",        # revenue - COGS - fees - shipping - other
    "gross_margin_pct",    # gross_profit / revenue * 100

    # demographics (count or share)
    # each group gets two columns: count and share_of_sales
] + [
    f"{grp}_count" for grp in DEMOGRAPHIC_GROUPS
] + [
    f"{grp}_share_pct" for grp in DEMOGRAPHIC_GROUPS
] + [
    # optional notes
    "notes",
    "data_source"          # e.g., POS, online order log, survey
]

# -------------------------------------------------------------------------
# 3) EXAMPLE ROWS
#    Replace placeholders or zeros with real numbers.
# -------------------------------------------------------------------------

example_rows_uk = [
    {
        "date": str(date(2026, 2, 1)),
        "country": "UK",
        "region_or_city": "London",
        "sku": "The Curse! Watermelon 50",
        "flavor": "Watermelon",
        "servings_per_tub": 50,
        "units_sold": 10,
        "revenue_local": 10 * 46.95,  # example: list price * units
        "cost_of_goods_sold": 10 * 25.00,  # placeholder COGS
        "shipping_cost": 10 * 4.95,       # if you charge or bear shipping
        "platform_fees": 10 * 2.00,       # placeholder marketplace fees
        "other_expenses": 50.0,
        # computed fields
        "gross_profit": None,
        "gross_margin_pct": None,
        # demographics placeholders
        "18-25_male_count": 2,
        "26-40_male_count": 1,
        "18-25_female_count": 3,
        "26-40_female_count": 4,
        # share placeholders (should sum to 100)
        "18-25_male_share_pct": None,
        "26-40_male_share_pct": None,
        "18-25_female_share_pct": None,
        "26-40_female_share_pct": None,
        "notes": "Sample row; replace with actual sales data.",
        "data_source": "POS placeholder"
    },
    # Add more rows as needed
]

example_rows_us = [
    {
        "date": str(date(2026, 2, 1)),
        "country": "US",
        "region_or_city": "CA",  # California example
        "sku": "The Curse! Watermelon 50",
        "flavor": "Watermelon",
        "servings_per_tub": 50,
        "units_sold": 8,
        "revenue_local": 8 * 28.49,
        "cost_of_goods_sold": 8 * 15.00,  # placeholder
        "shipping_cost": 8 * 3.50,
        "platform_fees": 8 * 1.50,
        "other_expenses": 40.0,
        "gross_profit": None,
        "gross_margin_pct": None,
        "18-25_male_count": 1,
        "26-40_male_count": 2,
        "18-25_female_count": 2,
        "26-40_female_count": 3,
        "18-25_male_share_pct": None,
        "26-40_male_share_pct": None,
        "18-25_female_share_pct": None,
        "26-40_female_share_pct": None,
        "notes": "Sample row; replace with actual sales data.",
        "data_source": "Online order log placeholder"
    },
    # Add more rows as needed
]

# -------------------------------------------------------------------------
# 4) UTILITY: compute derived fields for any row dict
# -------------------------------------------------------------------------
def compute_metrics(row):
    try:
        revenue = float(row["revenue_local"])
        cogs = float(row["cost_of_goods_sold"])
        shipping = float(row["shipping_cost"])
        fees = float(row["platform_fees"])
        other = float(row["other_expenses"])
    except Exception:
        # If data is missing or non-numeric, skip compute
        row["gross_profit"] = ""
        row["gross_margin_pct"] = ""
        return row

    gross_profit = revenue - cogs - shipping - fees - other
    if revenue != 0:
        margin_pct = gross_profit / revenue * 100
    else:
        margin_pct = 0

    row["gross_profit"] = round(gross_profit, 2)
    row["gross_margin_pct"] = round(margin_pct, 2)

    # Demographic share calculation
    # Count total sold from demographics if available
    total_demo = 0
    for grp in DEMOGRAPHIC_GROUPS:
        count_key = f"{grp}_count"
        try:
            total_demo += int(row.get(count_key, 0))
        except Exception:
            pass

    if total_demo > 0:
        for grp in DEMOGRAPHIC_GROUPS:
            count_key = f"{grp}_count"
            share_key = f"{grp}_share_pct"
            try:
                count = int(row.get(count_key, 0))
                row[share_key] = round(count / total_demo * 100, 1)
            except Exception:
                row[share_key] = ""
    else:
        for grp in DEMOGRAPHIC_GROUPS:
            row[f"{grp}_share_pct"] = ""

    return row

# Update example rows with computed metrics
example_rows_uk = [compute_metrics(r) for r in example_rows_uk]
example_rows_us = [compute_metrics(r) for r in example_rows_us]

# -------------------------------------------------------------------------
# 5) WRITE CSV FILES
# -------------------------------------------------------------------------
def write_csv(filename, rows):
    with open(filename, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELD_NAMES)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)

write_csv("jnx_sales_uk.csv", example_rows_uk)
write_csv("jnx_sales_us.csv", example_rows_us)

print("CSV files written: jnx_sales_uk.csv, jnx_sales_us.csv")

CSV files written: jnx_sales_uk.csv, jnx_sales_us.csv
